# Feature Engineering V2 — Prompt Injection Detection

This notebook generates the V2 feature dataset: 19 hand-crafted features + 384-dim semantic embeddings.

In [5]:
import os
import sys
import time
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))
from src_v2.features import PromptInjectionFeatureEngineerV2, FEATURE_NAMES_V2
from src_v2.embeddings import EmbeddingGenerator, EMBEDDING_DIM

print(f"Hand-crafted features: {len(FEATURE_NAMES_V2)}")
print(f"Embedding dimensions: {EMBEDDING_DIM}")
print(f"Total features: {len(FEATURE_NAMES_V2) + EMBEDDING_DIM}")

Hand-crafted features: 19
Embedding dimensions: 384
Total features: 403


In [6]:
CONFIG = {
    'data_path':  '/home/linkezio/Datasets/prompt_injection_dataset.csv',
    'output_path': '../data/features_engineered_v2.csv',
    'embedding_batch_size': 64,
    'hf_dataset': 'walledai/PromptInject',
}

In [7]:
df = pd.read_csv(CONFIG['data_path'])
if 'prompt' in df.columns and 'text' not in df.columns:
    df = df.rename(columns={'prompt': 'text'})
df['label'] = df['label'].astype(int)
print(f"Dataset: {df.shape[0]:,} rows")
print(f"Class distribution:{df['label'].value_counts()}")

Dataset: 399,741 rows
Class distribution:label
0    203067
1    196674
Name: count, dtype: int64


In [8]:
# --- QUICK TEST CELL ---
# Test embeddings on a tiny sample (including NaN edge cases)
test_df = df.head(100).copy()

# Inject some problematic values to verify the fix
test_df.loc[0, 'text'] = None
test_df.loc[1, 'text'] = float('nan')
test_df.loc[2, 'text'] = ''

print(f"Test sample: {len(test_df)} rows")
print("Text types:", test_df['text'].apply(type).value_counts().to_dict())

gen = EmbeddingGenerator()
texts = test_df['text'].astype(str).tolist()

embeddings = gen.encode_batch(
    texts,
    batch_size=8,
    show_progress_bar=False,
)

print(f"✓ Embeddings generated: {len(embeddings)} x {len(embeddings[0])}")
print("✓ NaN handling works correctly!")

Test sample: 100 rows
Text types: {<class 'str'>: 98, <class 'float'>: 2}


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✓ Embeddings generated: 100 x 384
✓ NaN handling works correctly!


In [9]:
fe = PromptInjectionFeatureEngineerV2()
start = time.time()
features_list = []
for idx, text in enumerate(df['text']):
    features_list.append(fe.extract_all(str(text)))
    if (idx + 1) % 50000 == 0:
        elapsed = time.time() - start
        print(f"  Processed: {idx+1}/{len(df)} | Elapsed: {elapsed:.1f}s")

features_df = pd.DataFrame(features_list, columns=FEATURE_NAMES_V2)
elapsed = time.time() - start
print(f"\nHand-crafted features extracted: {features_df.shape}")
print(f"Time: {elapsed:.1f}s ({elapsed/len(df)*1000:.2f}ms per text)")

  Processed: 50000/399741 | Elapsed: 10.8s
  Processed: 100000/399741 | Elapsed: 21.4s
  Processed: 150000/399741 | Elapsed: 32.2s
  Processed: 200000/399741 | Elapsed: 43.0s
  Processed: 250000/399741 | Elapsed: 53.8s
  Processed: 300000/399741 | Elapsed: 64.9s
  Processed: 350000/399741 | Elapsed: 75.5s

Hand-crafted features extracted: (399741, 19)
Time: 86.7s (0.22ms per text)


In [10]:
gen = EmbeddingGenerator()
texts = df['text'].astype(str).tolist()

start = time.time()
embeddings = gen.encode_batch(
    texts,
    batch_size=CONFIG['embedding_batch_size'],
    show_progress_bar=True,
)
elapsed = time.time() - start

embedding_cols = [f"embedding_{i}" for i in range(EMBEDDING_DIM)]
embeddings_df = pd.DataFrame(embeddings, columns=embedding_cols)
print(f"\nEmbeddings generated: {embeddings_df.shape}")
print(f"Time: {elapsed:.1f}s ({elapsed/len(df)*1000:.2f}ms per text)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/6246 [00:00<?, ?it/s]


Embeddings generated: (399741, 384)
Time: 144.2s (0.36ms per text)


In [11]:
combined_df = pd.concat([features_df, embeddings_df], axis=1)
combined_df['label'] = df['label'].values

median_ld = combined_df['lexical_diversity'].median()
combined_df['lexical_diversity'] = combined_df['lexical_diversity'].fillna(median_ld)
nulls = combined_df.isnull().sum()
remaining_nulls = nulls[nulls > 0]
if len(remaining_nulls) > 0:
    print("WARNING: Remaining nulls:")
    print(remaining_nulls)
else:
    print("Zero nulls after preprocessing")

print(f"\nFinal dataset: {combined_df.shape}")
print(f"Features: {combined_df.shape[1] - 1}")
print(f"Memory: {combined_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Zero nulls after preprocessing

Final dataset: (399741, 404)
Features: 403
Memory: 1232.1 MB


In [12]:
os.makedirs('data', exist_ok=True)
combined_df.to_csv(CONFIG['output_path'], index=False)
file_size = os.path.getsize(CONFIG['output_path']) / 1024**2
print(f"Saved to {CONFIG['output_path']}")
print(f"File size: {file_size:.1f} MB")
print(f"Shape: {combined_df.shape}")

Saved to ../data/features_engineered_v2.csv
File size: 3147.1 MB
Shape: (399741, 404)
